# Ruby pressure — batch fitting

Fits R1/R2 (two Pearson VII peaks + cubic background) for every spectrum in a folder with `rubyfit.py`,
then computes the pressure from the R1 shift of the sample ("inside") rubies relative to the reference
("outside") rubies **at the same temperature**.

File names give the conditions, e.g. `ice_VI_H2O_inside_297K_ruby3 044.csv` → sample, 297 K, ruby 3.
Repeat spectra of the same ruby are averaged first; fits that fail the quality checks are listed and excluded.

The same thing from a terminal: `python rubyfit.py spectra -o rubyfit_results` (see `python rubyfit.py --help`).

In [ ]:
%matplotlib inline
import time
import pandas as pd
import rubyfit

# --- settings -------------------------------------------------------------------------
DATA = ["spectra"]          # folders, files or glob patterns, e.g. ["spectra/*297K*.csv"]
SCALE = "mao1986"             # or "shen2020"
REF_WAVELENGTH = None         # e.g. 694.22 for single-ruby mode (no reference rubies)
WINDOW = (-4.5, 4.0)          # fit window around R1 in nm; None = full spectrum (as the old notebook)
PROFILE = "pearson7"          # or "split_pearson7": asymmetric peaks (R1 = peak maximum)
JOBS = 1                      # >1 to fit in parallel (only worth it for hundreds of files or with plots)
OUT = "rubyfit_results"       # where CSV files and plots are saved

In [ ]:
paths = rubyfit.find_files(DATA)
t0 = time.perf_counter()
fits = rubyfit.fit_files(paths, jobs=JOBS, window=WINDOW, profile=PROFILE)
fits, per_ruby, summary = rubyfit.pressure_tables(fits, scale=SCALE, ref_wavelength=REF_WAVELENGTH)
print(f"{len(fits)} spectra fitted in {time.perf_counter() - t0:.2f} s, "
      f"{(~fits.ok).sum()} flagged")
fits.loc[~fits.ok, ["file", "flags"]]

## Pressure per temperature
`P_err` combines the scatter between sample rubies and between reference rubies; `P_ruby_std` is the spread between sample rubies (pressure gradient + noise).

In [ ]:
view = summary.assign(P_MPa=1000 * summary.P_GPa, P_err_MPa=1000 * summary.P_err_GPa,
                      P_ruby_std_MPa=1000 * summary.P_ruby_std_GPa)
view[["temperature_K", "n_reference_rubies", "lambda0_nm", "lambda0_err_nm", "n_sample_rubies",
      "r1_sample_nm", "P_MPa", "P_err_MPa", "P_ruby_std_MPa"]].round(4)

In [ ]:
rubyfit.plot_summary(per_ruby, summary)

## Cross-check and error budget
Pressure of every reference × sample ruby pairing (rows: reference rubies, columns: sample rubies), then the uncertainty budget and the spread of the fits vs temperature and pressure. `outlier` in `per_ruby` flags rubies outside the 99% range of the other rubies on the same side.

In [ ]:
cross = rubyfit.cross_matrix(per_ruby if REF_WAVELENGTH is None else per_ruby.iloc[:0], SCALE)
budget = rubyfit.error_budget(fits, per_ruby, summary, cross, SCALE, REF_WAVELENGTH)
per_ruby.loc[per_ruby.outlier, ["temperature_K", "side", "ruby", "r1_dev_from_others_nm", "outlier_t", "outlier_t_limit"]]

In [ ]:
rubyfit.plot_cross(cross, per_ruby) if len(cross) else 'no reference rubies'

In [ ]:
rubyfit.plot_error_budget(cross, budget)

In [ ]:
rubyfit.plot_spread(budget)

## Per ruby

In [ ]:
per_ruby.assign(P_MPa=1000 * per_ruby.P_GPa, P_err_MPa=1000 * per_ruby.P_err_GPa)[
    ["temperature_K", "side", "ruby", "n_spectra", "r1_mean_nm", "r1_err_nm", "P_MPa", "P_err_MPa"]].round(4)

## Inspect one fit
Change the file name to look at any spectrum (flagged ones first).

In [ ]:
name = fits.sort_values("ok").file.iloc[0]
x, y = rubyfit.read_spectrum(fits.set_index("file").loc[name, "path"])
fit = rubyfit.fit_spectrum(x, y, window=WINDOW, profile=PROFILE)
rubyfit.plot_fit(fit, title=name)

## Save

In [ ]:
import os
os.makedirs(OUT, exist_ok=True)
fits.drop(columns="path").to_csv(os.path.join(OUT, "fits.csv"), index=False)
per_ruby.to_csv(os.path.join(OUT, "pressure_by_ruby.csv"), index=False)
summary.to_csv(os.path.join(OUT, "pressure_summary.csv"), index=False)
budget.to_csv(os.path.join(OUT, "error_budget.csv"), index=False)
rubyfit.plot_summary(per_ruby, summary, os.path.join(OUT, "summary.png"))
rubyfit.plot_error_budget(cross, budget, os.path.join(OUT, "error_budget.png"))
rubyfit.plot_spread(budget, os.path.join(OUT, "spread_vs_T_P.png"))
if len(cross):
    cross.to_csv(os.path.join(OUT, "pressure_cross.csv"), index=False)
    rubyfit.plot_cross(cross, per_ruby, os.path.join(OUT, "cross_matrix.png"))
print("saved to", os.path.abspath(OUT))